# 중간고사 대체 실습과제 — 문제 2

## SWOT 정량화와 TOWS 전략

| 항목 | 내용 |
|------|------|
| **관련 챕터** | Ch02. SWOT 분석과 PESTLE 분석 |
| **핵심 개념** | SWOT 정량화(중요도×평가), PESTLE, TOWS 매트릭스 |
| **배점** | 25점 |
| **제출** | 이 노트북(.ipynb)을 실행 결과 포함하여 제출 |

---

## 시나리오

건강 도시락 구독 서비스 **"프레시밀"**이 투자 유치를 준비하고 있습니다.
전략기획팀이 사전 조사를 통해 정리한 SWOT 요인과 PESTLE 이슈 데이터가 제공됩니다.
이를 정량화하고 전략을 도출하세요.

In [1]:
# 환경설정
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [6]:
# 데이터 로드
swot = pd.read_csv('data/swot_freshmeal.csv')
pestle = pd.read_csv('data/pestle_freshmeal.csv')

print('=== SWOT 데이터 (Category, Factor, Weight, Rating) ===')
print(swot.to_string(index=False))
print(f'\n=== PESTLE 데이터 (Factor, Category, Issue, Impact_Score) ===')
print(pestle.to_string(index=False))

=== SWOT 데이터 (Category, Factor, Weight, Rating) ===
Category              Factor  Weight  Rating
       S      독자적 AI 추천 알고리즘       5       5
       S    1만명 이상 회원 데이터 보유       4       4
       S         직관적 앱 UI/UX       4       4
       S  빠른 개발 주기 (2주 스프린트)       3       3
       W           마케팅 예산 부족       5       5
       W  배송 인프라 미보유 (외주 의존)       4       4
       W       CS 인력 부족 (2명)       4       3
       W         식품 안전 인증 미비       3       4
       O     건강식 시장 연 20% 성장       5       4
       O  1인 가구 급증 (전체의 34%)       4       4
       O ESG 경영 트렌드 (친환경 포장)       3       3
       O     정부 식품 스타트업 지원사업       3       3
       T     대형 유통사 구독서비스 진출       5       5
       T   원재료 가격 상승 (연 15%)       4       4
       T    배달앱의 건강식 카테고리 확대       4       3
       T          개인정보보호법 강화       3       2

=== PESTLE 데이터 (Factor, Category, Issue, Impact_Score) ===
Factor Category                         Issue  Impact_Score
     P       정치    식품 위생 규제 강화 (HACCP 의무화 확대)             4
   

---

## 과제 1. SWOT 정량화 (10점)

### 요구사항

1. **(3점)** SWOT 데이터에 **가중 점수(Weighted = Weight × Rating)** 열을 추가하고, **카테고리(S/W/O/T)별 총점**을 계산하여 출력하세요.

2. **(4점)** 아래 시각화를 만드세요:
   - S/W/O/T 각 카테고리의 **요인별 가중 점수** 수평 막대 차트 (2×2 서브플롯)

3. **(3점)** 결과를 해석하세요:
   - 가장 점수가 높은 강점(S)은? 가장 심각한 위협(T)은?
   - S총점 vs W총점, O총점 vs T총점을 비교하여 프레시밀의 전략적 위치를 1문장으로 요약

In [7]:
# ✏️ 과제 1-1: 가중 점수 계산 + 카테고리별 총점 출력
swot['Weighted'] = swot['Weight'] * swot['Rating']
total_scores = swot.groupby('Category')['Weighted'].sum()

print("카테고리별 총점:")
for cat, score in total_scores.items():
    print(f"{cat}: {score}")

카테고리별 총점:
O: 54
S: 66
T: 59
W: 65


In [9]:
# ✏️ 과제 1-2: 카테고리별 요인 가중 점수 수평 막대 차트 (2×2 서브플롯)
fig = make_subplots(rows=2, cols=2, subplot_titles=['강점 (S)', '약점 (W)', '기회 (O)', '위협 (T)'])

categories = ['S', 'W', 'O', 'T']
positions = [(1,1), (1,2), (2,1), (2,2)]

for cat, pos in zip(categories, positions):
    subset = swot[swot['Category'] == cat]
    fig.add_trace(go.Bar(
        x=subset['Weighted'],
        y=subset['Factor'],
        orientation='h',
        name=cat,
        marker=dict(color='white', line=dict(color='black', width=1)),
        showlegend=False
    ), row=pos[0], col=pos[1])

fig.update_layout(
    title='SWOT 카테고리별 요인 가중 점수',
    template='plotly_white',
    font=dict(color='black')
)
fig.show()

### ✏️ 과제 1-3: 해석 (이 셀에 직접 작성)

- 가장 높은 강점: 독자적 AI 추천 알고리즘 
- 가장 심각한 위협: 대형 유통사 구독서비스 진출
- 전략적 위치 요약: 프레시밀은 내부 강점과 약점이 균형적이지만 외부적으로는 불리한 위치에 있으므로 차별화 전략이 필요함

---

## 과제 2. PESTLE 분석 (7점)

### 요구사항

1. **(4점)** PESTLE 데이터의 6개 카테고리별 평균 Impact_Score를 계산하고, **레이더 차트(Radar Chart)**로 시각화하세요.
   - 6개 축: P(정치), E(경제), S(사회), T(기술), L(법률), E_env(환경)

2. **(3점)** PESTLE에서 Impact가 가장 높은 카테고리 2개를 찾고, 이것이 프레시밀의 SWOT에서 어떤 **기회(O)** 또는 **위협(T)**으로 연결되는지 1~2문장으로 설명하세요.

In [14]:
# ✏️ 과제 2-1: PESTLE 카테고리별 평균 + 레이더 차트
import plotly.graph_objects as go

# PESTLE 데이터
data = [
    ("P",4),("P",3),
    ("E",5),("E",4),
    ("L",4),("L",3),
    ("E_env",4),("E_env",3)
]

categories = ["P","E","S","T","L","E_env"]

# 평균 계산
avg = {}
for c in categories:
    vals = [v for k,v in data if k == c]
    avg[c] = sum(vals)/len(vals) if vals else 0

values = list(avg.values())

# 닫기
values += values[:1]
categories += categories[:1]

# 레이더 차트
fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=values,
    theta=categories,
    mode='lines+markers',
    line=dict(color='black', width=2),
    marker=dict(
        color='white',
        size=8,
        line=dict(color='black', width=2)
    ),
    fill='toself',
    fillcolor='rgba(0,0,0,0.05)',
    name='PESTLE'
))

fig.update_layout(
    title='PESTLE 카테고리별 평균 Impact Score',
    polar=dict(
        bgcolor='white',
        radialaxis=dict(
            visible=True,
            range=[0,5],   # 핵심!
            gridcolor='gray',
            tickfont=dict(color='black')
        ),
        angularaxis=dict(
            tickfont=dict(color='black')
        )
    ),
    showlegend=False,
    template='plotly_white',
    font=dict(color='black')
)

fig.show()



### ✏️ 과제 2-2: PESTLE → SWOT 연결 (이 셀에 직접 작성)

- Impact 가장 높은 카테고리 2개: (E경제, P정치가 가장 높다)
- SWOT 연결 설명: (경제(E)의 높은 영향력은 원재료 가격 상승이라는 위협(T)을 강화한다. 그리고 정치(P)의 규제 강화는 식품 안전 인증 미비라는 약점(W)을 더욱 부각시킨다.)

---

## 과제 3. TOWS 전략 도출 (8점)

| | **기회(O)** | **위협(T)** |
|------|------------|------------|
| **강점(S)** | **SO 전략** (강점으로 기회 활용) | **ST 전략** (강점으로 위협 대응) |
| **약점(W)** | **WO 전략** (약점 보완하여 기회 활용) | **WT 전략** (약점·위협 최소화) |

### 요구사항

1. **(8점)** 4가지 전략(SO, ST, WO, WT)을 각 1개씩 작성하세요.
   - 각 전략에서 **SWOT의 구체적 요인**(과제 1 데이터)을 인용하세요
   - 예시 형식: "[AI 추천 알고리즘(S)]을 활용하여 [건강식 시장 성장(O)]을 공략 → 개인 맞춤 추천 고도화"

### ✏️ 과제 3 답안 (이 셀에 직접 작성)

**SO 전략 (강점 × 기회):**

(독자적 AI 추천 알고리즘(S)을 활용해 건강식 시장 연 20% 성장(O)을 공략한다. 그럼 개인 맞춤형 건강식 구독 서비스가 고도화 될 것이다.)

**ST 전략 (강점 × 위협):**

(1만명 이상 회원 데이터(S)를 기반으로 대형 유통사 구독서비스 진출(T)에 대응한다. 그럼 고객 데이터 기반 고객 유지 전략이 강화됨.)

**WO 전략 (약점 × 기회):**

(마케팅 예산 부족(W)을 보완하기 위해 1인 가구 급증(O) 타겟 SNS 바이럴 마케팅에 집중한다. 그럼 저비용 고효율 고객을 확보할 수 있다)

**WT 전략 (약점 × 위협):**

(식품 안전 인증 미비(W)를 개선하여 식품 위생 규제 강화(T)에 대응한다. 그럼 식품 안전 인증 기반으로 리스크 관리가 강화됨.)

---

## 채점 기준

| 과제 | 배점 | 채점 포인트 |
|------|------|------------|
| 과제 1. SWOT 정량화 | 10점 | 가중점수(3) + 시각화(4) + 해석(3) |
| 과제 2. PESTLE 분석 | 7점 | 레이더차트(4) + SWOT 연결(3) |
| 과제 3. TOWS 전략 | 8점 | 4가지 전략 각 2점 |
| **합계** | **25점** | |

---

## 참고: 예상 정답값

**과제 1 카테고리별 총점:**

| 카테고리 | 총점 |
|---------|------|
| S (강점) | **66** |
| W (약점) | **65** |
| O (기회) | **54** |
| T (위협) | **59** |

- 가장 높은 강점: AI 추천 알고리즘 (25점)
- 가장 심각한 위협: 대형 유통사 진출 (25점)
- S(66) ≈ W(65), O(54) < T(59) → 내부는 균형, 외부 위협이 기회보다 약간 큼

**과제 2 PESTLE 평균 Impact:**

| 카테고리 | 평균 |
|---------|------|
| P | 3.5 | E | 4.5 | S | 4.0 | T | 4.5 | L | 3.5 | E_env | 3.5 |

- 경제(E)와 기술(T)이 4.5로 가장 높음